# Product Pricer — Fine-Tuning Learning Curve
### Week 6 Exercise — Stella Oiro (Andela AI Engineering Bootcamp)

Fine-tune GPT-4.1-nano to predict product prices from descriptions.

**The personalisation:** instead of a single fine-tuning run, this notebook trains at
four different dataset sizes — 50, 200, 1000, 5000 — and plots a **learning curve** showing
how much accuracy improves as training data grows.

**Pipeline:**
1. Load Ed's pre-processed Amazon dataset (`ed-donner/items_lite`) from HuggingFace
2. Submit 4 fine-tuning jobs at increasing training sizes
3. Evaluate all 4 fine-tuned models on the same test set
4. Plot error vs training size (the learning curve)
5. Gradio UI to try the best model live

**Estimated cost:** ~$1.50 total for all four fine-tuning runs

In [1]:
# Imports

import os
import re
import sys
import json
import time
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Give this notebook access to the week6 pricer module
sys.path.insert(0, "../../")
from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
openai = OpenAI()

api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-'):
    print(f"OpenAI API key found: {api_key[:12]}...")
else:
    print("OpenAI API key not found — check your .env file")

OpenAI API key found: sk-proj-Zcyb...


In [2]:
# Constants

BASE_MODEL    = "gpt-4.1-nano-2025-04-14"
TRAIN_SIZES   = [50, 200, 1000, 5000]   # the four fine-tuning experiments
EVAL_SIZE     = 200                      # test items to evaluate each model on

print(f"Base model:    {BASE_MODEL}")
print(f"Training sizes: {TRAIN_SIZES}")
print(f"Eval size:      {EVAL_SIZE} items per model")

Base model:    gpt-4.1-nano-2025-04-14
Training sizes: [50, 200, 1000, 5000]
Eval size:      200 items per model


## Step 1 — Load the dataset

We load Ed's pre-processed dataset from HuggingFace — no data curation cost.
Each `Item` has a `title`, `price`, `category`, `summary`, and `prompt`.

In [3]:
print("Loading dataset from HuggingFace (ed-donner/items_lite)...")
train, val, test = Item.from_hub("ed-donner/items_lite")

print(f"Train: {len(train):,} items")
print(f"Val:   {len(val):,} items")
print(f"Test:  {len(test):,} items")
print()
print("Sample item:")
print(train[0])
print()
print("Sample summary:")
print(train[0].summary)

Loading dataset from HuggingFace (ed-donner/items_lite)...


README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 20,000 items
Val:   1,000 items
Test:  1,000 items

Sample item:
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full=None weight=1.5 summary='Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.' prompt=None id=None

Sample summary:
Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  
Category: Home Hardware  
Brand: Schlage  
Description: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  
Details: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical

## Step 2 — Prepare training data as JSONL

OpenAI fine-tuning expects JSONL format — one JSON object per line,
each containing a `messages` list with the user question and correct assistant answer.

We use the `Item.test_prompt()` method which strips the answer from the full prompt
so the model doesn't see the price in the user message.

In [ ]:
def messages_for(item, include_answer=True):
    """Format one Item as a fine-tuning training pair."""
    messages = [
        {"role": "system", "content": "You estimate prices of products. Reply only with the price, no explanation."},
        {"role": "user",   "content": f"Estimate the price of this product.\n\n{item.summary}"},
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": f"Price is ${item.price:.2f}"})
    return messages


def write_jsonl(items, filepath):
    """Write items to a JSONL file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w") as f:
        for item in items:
            f.write(json.dumps({"messages": messages_for(item)}) + "\n")


# Preview one training example
print(json.dumps({"messages": messages_for(train[0])}, indent=2))

## Step 3 — Submit all four fine-tuning jobs

We submit one job per training size and store the job IDs.
Jobs run asynchronously — typically 15-60 minutes depending on size.

**Approximate costs:**
| Training size | Estimated cost |
|---|---|
| 50 | ~$0.01 |
| 200 | ~$0.04 |
| 1,000 | ~$0.20 |
| 5,000 | ~$1.00 |
| **Total** | **~$1.25** |

In [ ]:
job_ids = {}

for size in TRAIN_SIZES:
    train_path = f"jsonl/train_{size}.jsonl"
    val_path   = f"jsonl/val_{size}.jsonl"

    # Write JSONL files
    write_jsonl(train[:size], train_path)
    write_jsonl(val[:50],     val_path)       # 50 val items for all runs

    # Upload to OpenAI
    with open(train_path, "rb") as f:
        train_file = openai.files.create(file=f, purpose="fine-tune")
    with open(val_path, "rb") as f:
        val_file = openai.files.create(file=f, purpose="fine-tune")

    # Submit fine-tuning job
    job = openai.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters={"n_epochs": 1, "batch_size": 1},
        suffix=f"pricer-{size}"
    )

    job_ids[size] = job.id
    print(f"  size={size:>5,}  job_id={job.id}")

print()
print(f"All {len(job_ids)} jobs submitted. Check status at:")
print("  https://platform.openai.com/finetune")

## Step 4 — Wait for all jobs to complete

Run this cell to poll status every 60 seconds until all jobs are done.
Alternatively, check https://platform.openai.com/finetune and run the next cell when ready.

Typical wait: 15 min (50 examples) to 60 min (5000 examples).

In [ ]:
def wait_for_jobs(job_ids, poll_seconds=60):
    """Poll until all fine-tuning jobs are succeeded or failed."""
    pending = set(job_ids.values())
    while pending:
        time.sleep(poll_seconds)
        still_pending = set()
        for job_id in pending:
            status = openai.fine_tuning.jobs.retrieve(job_id).status
            if status in ("succeeded", "failed", "cancelled"):
                label = [k for k, v in job_ids.items() if v == job_id][0]
                print(f"  size={label:>5,}  status={status}")
            else:
                still_pending.add(job_id)
        pending = still_pending
        if pending:
            print(f"  {len(pending)} job(s) still running... checking again in {poll_seconds}s")
    print("All jobs complete.")


wait_for_jobs(job_ids)

In [ ]:
# Collect the fine-tuned model names

model_names = {}
for size, job_id in job_ids.items():
    job = openai.fine_tuning.jobs.retrieve(job_id)
    model_names[size] = job.fine_tuned_model
    print(f"  size={size:>5,}  model={job.fine_tuned_model}")

## Step 5 — Evaluate all four models

Each model is evaluated on the same 200 test items.
We record the average absolute error (in dollars) for each training size.

In [ ]:
import sys
sys.path.insert(0, "../../")
from pricer.evaluator import Tester


def get_price(response_text):
    """Extract a dollar amount from the model's response."""
    s = response_text.replace('$', '').replace(',', '')
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0.0


def make_predictor(model_name, size):
    """Return a predictor function for the given fine-tuned model."""
    def predictor(item):
        response = openai.chat.completions.create(
            model=model_name,
            messages=messages_for(item, include_answer=False),
            max_tokens=7
        )
        return get_price(response.choices[0].message.content)
    predictor.__name__ = f"fine_tuned_{size}"
    return predictor


print("Evaluating each fine-tuned model on the test set...")
print("(This will make API calls — takes a few minutes per model)\n")

results = {}   # size -> average_error

for size, model_name in model_names.items():
    print(f"--- Training size: {size:,} ---")
    predictor = make_predictor(model_name, size)
    tester = Tester(predictor, test, title=f"Fine-tuned {size:,} examples", size=EVAL_SIZE)
    tester.run()
    results[size] = sum(tester.errors) / len(tester.errors)
    print(f"  Average error: ${results[size]:.2f}\n")

## Step 6 — Plot the learning curve

How much does accuracy improve as we add more training data?
The learning curve shows error vs training size on a log scale.

In [ ]:
sizes  = list(results.keys())
errors = list(results.values())

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(sizes, errors, marker='o', linewidth=2.5, markersize=8, color='steelblue')

for x, y in zip(sizes, errors):
    ax.annotate(f"${y:.1f}", (x, y), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)

ax.set_xscale('log')
ax.set_xticks(sizes)
ax.set_xticklabels([str(s) for s in sizes])
ax.set_xlabel("Training examples", fontsize=12)
ax.set_ylabel("Average absolute error ($)", fontsize=12)
ax.set_title("Fine-Tuning Learning Curve — GPT-4.1-nano Product Pricer", fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_size = min(results, key=results.get)
print(f"Best model: trained on {best_size:,} examples with ${results[best_size]:.2f} average error")

## Step 7 — Gradio UI

Use the best fine-tuned model interactively.
Paste any product description and get an estimated price.

In [ ]:
best_model = model_names[best_size]


def predict_price(description):
    """Predict the price of a product from its description."""
    if not description.strip():
        return "Please enter a product description."
    response = openai.chat.completions.create(
        model=best_model,
        messages=[
            {"role": "system", "content": "You estimate prices of products. Reply only with the price, no explanation."},
            {"role": "user",   "content": f"Estimate the price of this product.\n\n{description}"},
        ],
        max_tokens=7
    )
    price = get_price(response.choices[0].message.content)
    return f"${price:.2f}"


EXAMPLES = [
    "Title: Wireless Noise-Cancelling Headphones\nCategory: Electronics\nBrand: Sony\nDescription: Over-ear Bluetooth headphones with 30hr battery and active noise cancellation.\nDetails: Foldable design, built-in microphone, USB-C charging.",
    "Title: Cordless Power Drill\nCategory: Tools and Home Improvement\nBrand: DeWalt\nDescription: 20V MAX lithium-ion drill with variable speed and LED work light.\nDetails: 2-speed transmission, 1/2 inch chuck, includes battery and charger.",
    "Title: Stainless Steel Cookware Set\nCategory: Appliances\nBrand: Cuisinart\nDescription: 12-piece tri-ply bonded cookware set with glass lids.\nDetails: Oven safe to 500F, dishwasher safe, induction compatible.",
    "Title: Kids Building Blocks Set\nCategory: Toys and Games\nBrand: Mega Bloks\nDescription: 80-piece large building blocks for toddlers aged 1-5.\nDetails: BPA-free, non-toxic, compatible with Duplo.",
]

with gr.Blocks(title="Product Pricer", theme=gr.themes.Soft()) as ui:
    gr.Markdown(f"""
    # Product Pricer
    Fine-tuned GPT-4.1-nano trained on {best_size:,} Amazon product descriptions.
    Paste a product description to get an estimated price.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            description = gr.Textbox(
                label="Product Description",
                placeholder="Title: ...\nCategory: ...\nBrand: ...\nDescription: ...\nDetails: ...",
                lines=7
            )
            submit = gr.Button("Estimate Price", variant="primary")

        with gr.Column(scale=1):
            output = gr.Textbox(label="Estimated Price", lines=2, interactive=False)

    gr.Examples(examples=EXAMPLES, inputs=description)
    submit.click(fn=predict_price, inputs=description, outputs=output)

print("UI built.")

In [ ]:
ui.launch(inbrowser=True)